# Image Enhancer — Colab GPU Worker

**Steps:**
1. `Runtime → Change runtime type → T4 GPU` → Save
2. `Runtime → Run all` (`Ctrl+F9`)
3. Wait for Cell 1 to finish (~2 min)
4. Cell 3 prints a URL like `https://xxxx.gradio.live`
5. Copy that URL and use it with `--remote-url` on your PC

> Keep this tab open. Cell 4 runs forever to prevent idle disconnection.

In [ ]:
# ── Cell 1: Check GPU + Install ───────────────────────────────────────────
import subprocess, sys

r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                    '--format=csv,noheader'], capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit('NO GPU — go to Runtime → Change runtime type → T4 GPU')
print(f'GPU: {r.stdout.strip()}')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'gradio==4.44.1', 'gradio-client', 'numpy<2', 'pillow',
    'opencv-python-headless', 'psutil'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'torch==2.2.0', 'torchvision==0.17.0',
    '--index-url', 'https://download.pytorch.org/whl/cu118'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'basicsr', 'realesrgan'])
print('All deps installed.')

In [ ]:
# ── Cell 2: Compatibility shims + Load model onto GPU ─────────────────────
import sys, types, os, time, urllib.request
from pathlib import Path
import numpy as np
import cv2
import torch
from PIL import Image

# Python 3.12 shim — basicsr still imports distutils.version
if 'distutils.version' not in sys.modules:
    try:
        from setuptools._distutils.version import LooseVersion
    except ImportError:
        from packaging.version import parse as LooseVersion
    dm = types.ModuleType('distutils'); vm = types.ModuleType('distutils.version')
    vm.LooseVersion = LooseVersion; dm.version = vm
    sys.modules['distutils'] = dm; sys.modules['distutils.version'] = vm

if 'torchvision.transforms.functional_tensor' not in sys.modules:
    try:
        import torchvision.transforms.functional as _tf
        sys.modules['torchvision.transforms.functional_tensor'] = _tf
    except ImportError: pass

from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer

assert torch.cuda.is_available(), 'CUDA not available'
device = torch.device('cuda:0')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

os.makedirs('models', exist_ok=True)
MODEL_PATH = 'models/RealESRGAN_x4plus.pth'
if not os.path.exists(MODEL_PATH):
    print('Downloading model (~64 MB)...')
    urllib.request.urlretrieve(
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
        MODEL_PATH)

# Load model — tile=0 means full image on GPU (fastest, uses full VRAM)
# Change tile=256 if you get CUDA out-of-memory errors
_net = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
               num_block=23, num_grow_ch=32, scale=4)
_upsampler = RealESRGANer(
    scale=4, model_path=MODEL_PATH, model=_net,
    tile=0,        # 0 = full image — uses all VRAM, fastest
    tile_pad=10, pre_pad=0,
    half=True,     # float16 — 2x faster, half VRAM
    device=device,
)
print(f'Model loaded on GPU (half={_upsampler.half})')
print(f'VRAM used: {torch.cuda.memory_allocated()/1024**3:.2f} GB')

In [ ]:
# ── Cell 3: Define API + Launch Gradio server ─────────────────────────────
import gradio as gr, json, psutil

def enhance(
    image: np.ndarray,
    method: str = 'realesrgan',
    scale: int = 4,
    tile: int = 0,
    face_enhance: bool = False,
    progress=gr.Progress(track_tqdm=True),
) -> np.ndarray:
    scale = int(scale); tile = int(tile); method = str(method).strip().lower()
    h, w = image.shape[:2]
    print(f'[Worker] method={method} scale={scale}x tile={tile} size={w}x{h}')
    t0 = time.time()

    vf = torch.cuda.mem_get_info()[0]/1024**3
    vt = torch.cuda.mem_get_info()[1]/1024**3
    print(f'[Worker] VRAM free: {vf:.1f}/{vt:.1f} GB')

    if method == 'realesrgan':
        # Update tile size if caller requested a specific one
        if tile != _upsampler.tile_size:
            _upsampler.tile_size = tile

        # Calculate tiles for progress
        if tile > 0:
            total_tiles = len(range(0,h,tile)) * len(range(0,w,tile))
        else:
            total_tiles = 1

        tc = {'n': 0}
        orig_model = _upsampler.model

        class _Wrap:
            def __call__(self, *a, **k):
                res = orig_model(*a, **k)
                tc['n'] += 1
                try: gpu_util = torch.cuda.utilization()
                except: gpu_util = 0
                progress(tc['n']/total_tiles, desc=json.dumps({
                    'current': tc['n'], 'total': total_tiles,
                    'dt': round(time.time()-t0, 2), 'cpu': gpu_util
                }))
                return res
            def __getattr__(self, n): return getattr(orig_model, n)

        try:
            _upsampler.model = _Wrap()
            bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
            out_bgr, _ = _upsampler.enhance(bgr, outscale=scale)
            result = cv2.cvtColor(out_bgr, cv2.COLOR_BGR2RGB)
        finally:
            _upsampler.model = orig_model
            torch.cuda.empty_cache()

    elif method in ('bicubic', 'lanczos'):
        pil = Image.fromarray(image)
        rs = Image.BICUBIC if method == 'bicubic' else Image.LANCZOS
        result = np.array(pil.resize((w*scale, h*scale), rs))
    else:
        raise ValueError(f'Unknown method: {method}')

    print(f'[Worker] Done in {time.time()-t0:.2f}s  out={result.shape[1]}x{result.shape[0]}')
    return result


with gr.Blocks(title='Image Enhancer GPU Worker') as demo:
    gr.Markdown('## Image Enhancer — Colab GPU Worker\nCalled automatically by your local app.')
    with gr.Row():
        with gr.Column():
            img_in    = gr.Image(type='numpy', label='Input')
            method_in = gr.Radio(['realesrgan','bicubic','lanczos'], value='realesrgan', label='Method')
            scale_in  = gr.Slider(2, 8, step=2, value=4, label='Scale')
            tile_in   = gr.Slider(0, 512, step=64, value=0, label='Tile (0=full GPU)')
            face_in   = gr.Checkbox(value=False, label='Face Enhance')
            gr.Button('Enhance', variant='primary').click(
                fn=enhance,
                inputs=[img_in, method_in, scale_in, tile_in, face_in],
                outputs=gr.Image(type='numpy', label='Output'),
                api_name='enhance'
            )

demo.queue(max_size=4)
demo.launch(share=True, debug=False, show_error=True,
            inline=False, prevent_thread_lock=True)
print('Server launched. Copy the gradio.live URL above.')

In [ ]:
# ── Cell 4: Keep-alive loop (runs forever — prevents idle disconnect) ──────
import time, datetime, torch, psutil

print('Keep-alive running. Stop this cell manually when done.')
tick = 0
while True:
    tick += 1
    _ = torch.zeros(1, device='cuda').sum()   # tiny GPU op = keeps CUDA alive
    vf = torch.cuda.mem_get_info()[0]/1024**3
    vt = torch.cuda.mem_get_info()[1]/1024**3
    ram = psutil.virtual_memory().percent
    now = datetime.datetime.now().strftime('%H:%M:%S')
    print(f'[{now}] tick={tick:04d}  VRAM {vt-vf:.1f}/{vt:.1f}GB used  RAM {ram:.0f}%  server=running',
          flush=True)
    time.sleep(30)